# 01 — Data Acquisition: Download & Filter American Stories

This notebook downloads parquet files from the American Stories dataset for 1890–1896
and filters for articles related to the gold vs. silver monetary standard debate 
using Tier 1 keywords.

**Note**: We use `huggingface_hub` to download parquet files directly (rather than the
`datasets` library) to avoid Python 3.14 pickle compatibility issues.

**Output**: `data/american_stories/filtered_articles.parquet`

In [1]:
import sys
sys.path.insert(0, "..")

import pandas as pd
from src.data_utils import (
    TIER1_KEYWORDS,
    download_and_filter_year,
    download_and_filter_years,
)

/Users/cantstopkevin/venvs/jlab/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Keyword Review

Before streaming, let's review the Tier 1 keywords we're filtering on.

In [2]:
print(f"Number of Tier 1 keywords: {len(TIER1_KEYWORDS)}")
print("\nKeywords:")
for kw in TIER1_KEYWORDS:
    print(f"  - {kw}")

Number of Tier 1 keywords: 27

Keywords:
  - free silver
  - free coinage
  - gold standard
  - bimetallism
  - bimetallic
  - silver standard
  - sound money
  - honest money
  - sixteen to one
  - 16 to 1
  - crime of 73
  - crime of 1873
  - cross of gold
  - goldbugs
  - gold bugs
  - gold bug
  - silverites
  - silverite
  - sherman silver
  - bland-allison
  - bland allison
  - silver coinage
  - gold coinage
  - coinage of silver
  - coinage of gold
  - monetary standard
  - currency question


## Test Run: Single Year, Small Sample

First, let's test the pipeline on a small slice of one year (1893, the year of the
Panic and repeal of the Sherman Silver Purchase Act) to make sure everything works.

In [4]:
# Test with first 5,000 articles from 1893
df_test = download_and_filter_year(
    year="1893",
    max_articles=5000,
    min_article_length=100,
)

print(f"\nTest results: {len(df_test)} matching articles from first 5,000")
print(f"Columns: {list(df_test.columns)}")
df_test.head()

Finding parquet files for year 1893...
  Found 2 parquet file(s) for 1893.
  Loaded 656,805 total articles for 1893.
  Columns in parquet: ['article_id', 'newspaper_name', 'edition', 'date', 'page', 'headline', 'byline', 'article']
  Truncated to first 5,000 articles for testing.
  Extracted LCCN from article_id for 28 articles.
  Year 1893: 28 articles match keywords.

Test results: 28 matching articles from first 5,000
Columns: ['article_id', 'newspaper_name', 'edition', 'date', 'page', 'headline', 'byline', 'article', 'lccn', 'year', 'matched_keywords']


,article_id,newspaper_name,edition,date,page,headline,byline,article,lccn,year,matched_keywords
15,17_1893-07-19_p2_sn94052989_00175040262_189307...,The morning call.,01,1893-07-19,p2,SILVER STANDARD\n\nChamber of Commerce\n\n\nan...,,To the exclusion of a consideration of\nmany o...,sn94052989,1893,[coinage of silver]
568,12_1893-08-02_p1_sn95060905_00211107133_189308...,Tombstone epitaph.,01,1893-08-02,p1,BY WIRE.,,A telegram from Bankok indicates that the siam...,sn95060905,1893,"[gold standard, free silver, sherman silver]"
857,5_1893-08-09_p6_sn99021999_00206538892_1893080...,Omaha daily bee.,01,1893-08-09,p6,DIscmssed Tholr Plans.,,"WAsuINGToN, Aug S,-Immedately after\nThe readi...",sn99021999,1893,[free coinage]
887,69_1893-08-09_p6_sn99021999_00206538892_189308...,Omaha daily bee.,01,1893-08-09,p6,SENATOR HILLS BILL.\n\nHe Wants the sherman Ac...,,\n\n\nPledges the Country to Bimetaillsm.\n\n\...,sn99021999,1893,"[free coinage, bimetallism]"
1328,19_1893-05-04_p2_sn89051294_00393342808_189305...,The Green Forest tribune.,01,1893-05-04,p2,,,"SENATOR HANsBRoUaH, of North\nDakota, says: at...",sn89051294,1893,[free coinage]


In [5]:
# Inspect a sample article
if len(df_test) > 0:
    sample = df_test.iloc[0]
    print(f"Newspaper: {sample['newspaper_name']}")
    print(f"Date: {sample['date']}")
    print(f"Headline: {sample['headline']}")
    print(f"Matched keywords: {sample['matched_keywords']}")
    print(f"\nArticle text (first 500 chars):")
    print(sample['article'][:500])
else:
    print("No matches found in test sample. Consider increasing max_articles.")

Newspaper: The morning call.
Date: 1893-07-19
Headline: SILVER STANDARD

Chamber of Commerce


and Currency.

ITS VIEWS SENT TO CONGRESS

Some Resolutions on the Nicaragua
Canal, Hawaiian Annexation and


Other Important Matters.
Matched keywords: ['coinage of silver']

Article text (first 500 chars):
To the exclusion of a consideration of
many other matters of great importance
the Chamber of Commerce at its quarterly
meeting yesterday devoted much time to a
spirited discussion Of the silver question
and the currency.


The members earnestly debated the
great question that is just now the princl-
pal topic of discussion throughout the
financial world.


Many divergent lines of policy for the
Government to pursue were mapped out
and submitted for the Endorsement Of the
chamber before being bro


## Full Run: All Years 1890–1896

Now let's stream all seven years. This will take a while depending on your connection.

**Note**: If running on limited compute, you can set `max_articles_per_year` to cap
each year. Set to `None` for the full dataset.

In [6]:
YEARS = [str(y) for y in range(1890, 1897)]  # 1890 through 1896
SAVE_PATH = "../data/american_stories/filtered_articles_EXAMPLE.parquet"

# Set to None for full run, or an integer (e.g. 50000) for testing
MAX_ARTICLES_PER_YEAR = 100000

print(f"Years to process: {YEARS}")
print(f"Max articles per year: {MAX_ARTICLES_PER_YEAR or 'unlimited'}")

Years to process: ['1890', '1891', '1892', '1893', '1894', '1895', '1896']
Max articles per year: 100000


In [7]:
df_all = download_and_filter_years(
    years=YEARS,
    max_articles_per_year=MAX_ARTICLES_PER_YEAR,
    min_article_length=100,
    save_path=SAVE_PATH,
)

Finding parquet files for year 1890...
  Found 2 parquet file(s) for 1890.


  Loaded 540,615 total articles for 1890.
  Columns in parquet: ['article_id', 'newspaper_name', 'edition', 'date', 'page', 'headline', 'byline', 'article']
  Truncated to first 100,000 articles for testing.
  Extracted LCCN from article_id for 306 articles.
  Year 1890: 306 articles match keywords.
Finding parquet files for year 1891...
  Found 2 parquet file(s) for 1891.
  Loaded 620,461 total articles for 1891.
  Columns in parquet: ['article_id', 'newspaper_name', 'edition', 'date', 'page', 'headline', 'byline', 'article']
  Truncated to first 100,000 articles for testing.
  Extracted LCCN from article_id for 432 articles.
  Year 1891: 432 articles match keywords.
Finding parquet files for year 1892...
  Found 2 parquet file(s) for 1892.
  Loaded 527,044 total articles for 1892.
  Columns in parquet: ['article_id', 'newspaper_name', 'edition', 'date', 'page', 'headline', 'byline', 'article']
  Truncated to first 100,000 articles for testing.
  Extracted LCCN from article_id for 594

In [8]:
# Quick summary
print(f"Total filtered articles: {len(df_all):,}")
print(f"\nArticles by year:")
print(df_all['year'].value_counts().sort_index())
print(f"\nUnique newspapers: {df_all['newspaper_name'].nunique()}")
print(f"Unique LCCNs: {df_all['lccn'].nunique()}")

Total filtered articles: 6,179

Articles by year:
year
1890     306
1891     432
1892     594
1893     646
1894     424
1895    1100
1896    2677
Name: count, dtype: int64

Unique newspapers: 212
Unique LCCNs: 214


In [9]:
# Most common keywords
from collections import Counter

all_keywords = []
for kw_list in df_all['matched_keywords']:
    if isinstance(kw_list, list):
        all_keywords.extend(kw_list)
    elif isinstance(kw_list, str):
        all_keywords.extend(kw_list.split('|'))

kw_counts = Counter(all_keywords).most_common(20)
print("Most common matched keywords:")
for kw, count in kw_counts:
    print(f"  {kw}: {count:,}")

Most common matched keywords:
  free coinage: 2,141
  free silver: 1,720
  gold standard: 917
  sound money: 879
  coinage of silver: 772
  bimetallism: 370
  bimetallic: 258
  silver coinage: 246
  honest money: 224
  gold bug: 172
  16 to 1: 167
  gold bugs: 160
  currency question: 145
  silverites: 142
  silver standard: 116
  sherman silver: 112
  coinage of gold: 97
  goldbugs: 91
  sixteen to one: 57
  silverite: 35


## Verify Saved Data

Confirm the parquet file was saved correctly.

In [10]:
df_check = pd.read_parquet(SAVE_PATH)
print(f"Saved file has {len(df_check):,} rows and {len(df_check.columns)} columns")
print(f"File size: {pd.io.common.file_exists(SAVE_PATH)}")
df_check.info()

Saved file has 6,179 rows and 11 columns
File size: True
<class 'pandas.DataFrame'>
RangeIndex: 6179 entries, 0 to 6178
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   article_id        6179 non-null   str  
 1   newspaper_name    6179 non-null   str  
 2   edition           6179 non-null   str  
 3   date              6179 non-null   str  
 4   page              6179 non-null   str  
 5   headline          6179 non-null   str  
 6   byline            6179 non-null   str  
 7   article           6179 non-null   str  
 8   lccn              6179 non-null   str  
 9   year              6179 non-null   str  
 10  matched_keywords  6179 non-null   str  
dtypes: str(11)
memory usage: 14.4 MB
